In [1]:
# ================================================================
# POLY-FEVER — FINAL GEMMA 3:1B EXPERIMENT
# Based on the working Qwen/Kaggle methodology.
# ================================================================

import os
import re
import time
from pathlib import Path

import pandas as pd
import requests


# ================================================================
# 1. SETTINGS
# ================================================================

FOLDER = Path(
    r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research"
)

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:1b"

OUTPUT_FILE = (
    FOLDER / "Poly_FEVER_Gemma3_1B_QA_FINAL.csv"
)

print("Folder:", FOLDER)
print("Model:", MODEL)
print("Output:", OUTPUT_FILE)


# ================================================================
# 2. FIND ORIGINAL PAIRED POLY-FEVER DATASET
# ================================================================

candidates = []

for f in FOLDER.glob("*.csv"):
    name = f.name.lower()

    if (
        "poly" in name
        and "fever" in name
        and "paired" in name
        and "full_columns" in name
        and "gemma" not in name
        and "master" not in name
        and "recovered" not in name
        and "results" not in name
        and "qa_final" not in name
    ):
        candidates.append(f)

if not candidates:
    raise FileNotFoundError(
        "Original paired Poly-FEVER CSV not found."
    )

SOURCE_FILE = max(
    candidates,
    key=lambda p: p.stat().st_size
)

print("\nSOURCE:")
print(SOURCE_FILE)


# ================================================================
# 3. LOAD + VERIFY SOURCE
# ================================================================

df = pd.read_csv(SOURCE_FILE)

required = ["ID", "en", "Nepali", "Label"]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

df["ID_STR"] = (
    df["ID"]
    .astype(str)
    .str.strip()
)

df["GOLD"] = (
    df["Label"]
    .astype(str)
    .str.strip()
    .str.upper()
)

if df["ID_STR"].duplicated().any():
    raise ValueError("Duplicate source IDs found.")

if df["en"].isna().any():
    raise ValueError("Missing English statements found.")

if df["Nepali"].isna().any():
    raise ValueError("Missing Nepali statements found.")

if not set(df["GOLD"]).issubset({"TRUE", "FALSE"}):
    raise ValueError(
        f"Invalid Gold Label values: {set(df['GOLD'])}"
    )

print("\nSOURCE VERIFIED")
print("Rows:", len(df))
print("Expected evaluations:", len(df) * 2)


# ================================================================
# 4. EXPECTED PAIRS
# ================================================================

expected_pairs = set()

for item_id in df["ID_STR"]:
    expected_pairs.add((item_id, "English"))
    expected_pairs.add((item_id, "Nepali"))

print("Expected pairs:", len(expected_pairs))


# ================================================================
# 5. RESULT FILE
# ================================================================

if OUTPUT_FILE.exists():

    print("\nExisting clean result found.")
    print("Resuming from saved file.")

    results = pd.read_csv(
        OUTPUT_FILE
    )

else:

    print("\nStarting fresh.")

    results = pd.DataFrame(
        columns=[
            "ID",
            "Language",
            "Statement",
            "Reference Nepali",
            "Gold Label",
            "Gemma Raw Output",
            "Gemma Answer",
            "Verdict",
            "Error Type",
            "Strict Error"
        ]
    )


# Remove duplicate ID-language pairs if any.
results = results.drop_duplicates(
    subset=["ID", "Language"],
    keep="last"
).reset_index(drop=True)


# ================================================================
# 6. PAIR SET
# ================================================================

def get_pairs(data):
    if data.empty:
        return set()

    return {
        (
            str(row["ID"]).strip(),
            str(row["Language"]).strip()
        )
        for _, row in data.iterrows()
        if pd.notna(row["ID"])
        and pd.notna(row["Language"])
    }


# ================================================================
# 7. SAFE SAVE
# ================================================================

def safe_save(data):

    temp_file = OUTPUT_FILE.with_suffix(
        OUTPUT_FILE.suffix + ".tmp"
    )

    while True:

        try:

            data.to_csv(
                temp_file,
                index=False,
                encoding="utf-8-sig"
            )

            os.replace(
                temp_file,
                OUTPUT_FILE
            )

            return

        except PermissionError:

            print(
                "\nCSV is locked."
            )
            print(
                "Close it in Excel and leave it closed."
            )
            time.sleep(5)

        except Exception as e:

            print(
                "Save error:",
                e
            )
            time.sleep(5)


safe_save(results)


# ================================================================
# 8. OLLAMA CHECK
# ================================================================

def wait_for_ollama():

    while True:

        try:

            r = requests.get(
                "http://localhost:11434/api/tags",
                timeout=10
            )

            if r.status_code == 200:
                print("\n✅ Ollama online.")
                return

        except Exception:
            pass

        print(
            "Ollama unavailable. "
            "Retrying in 10 seconds..."
        )

        time.sleep(10)


wait_for_ollama()


# ================================================================
# 9. SOURCE LOOKUP
# ================================================================

source_lookup = {
    str(row["ID_STR"]): row
    for _, row in df.iterrows()
}


# ================================================================
# 10. GEMMA REQUEST
# ================================================================

def ask_gemma(prompt):

    retry = 0
    wait = 3

    while True:

        try:

            r = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt,
                    "stream": False,
                    "keep_alive": "5m",
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=(15, 600)
            )

            if r.status_code == 200:

                text = r.json().get(
                    "response",
                    ""
                ).strip()

                if text:
                    return text

                print("Empty Gemma response.")

            else:

                print(
                    "Ollama HTTP error:",
                    r.status_code
                )

        except requests.exceptions.ConnectionError:
            print("Ollama connection lost.")

        except requests.exceptions.Timeout:
            print("Ollama timeout.")

        except Exception as e:
            print("Ollama error:", e)

        retry += 1

        print(
            f"Retry {retry} in {wait}s..."
        )

        time.sleep(wait)

        wait = min(
            wait * 2,
            30
        )


# ================================================================
# 11. NORMALIZE GEMMA OUTPUT
# ================================================================
# Based on the Qwen reference:
# exact TRUE/FALSE first, then unambiguous occurrence.
# Otherwise UNKNOWN/NON_STANDARD.
# ================================================================

def normalize_prediction(text):

    text = str(text).strip().upper()

    if text == "TRUE":
        return "TRUE"

    if text == "FALSE":
        return "FALSE"

    first_word = (
        text.split()[0]
        if text.split()
        else ""
    )

    if first_word == "TRUE":
        return "TRUE"

    if first_word == "FALSE":
        return "FALSE"

    true_found = bool(
        re.search(r"\bTRUE\b", text)
    )

    false_found = bool(
        re.search(r"\bFALSE\b", text)
    )

    if true_found and not false_found:
        return "TRUE"

    if false_found and not true_found:
        return "FALSE"

    return "NON_STANDARD"


# ================================================================
# 12. GEMMA PROMPTS
# ================================================================
# IMPORTANT:
# There is NO dataset chunk in the prompt.
# This matches the working Qwen benchmark methodology.
# ================================================================

def build_english_prompt(statement):

    return f"""
You are a factual verification model.

Classify the following statement using your existing knowledge.

Return exactly one label:

TRUE
or
FALSE

Do not explain your answer.
Do not output anything except TRUE or FALSE.

Statement:
{statement}

Answer:
"""


def build_nepali_prompt(statement):

    return f"""
तपाईं एउटा तथ्यात्मक कथनको सत्यता जाँच गर्ने मोडेल हुनुहुन्छ।

तलको कथन TRUE वा FALSE भनेर वर्गीकरण गर्नुहोस्।

ठ्याक्कै एउटा मात्र उत्तर दिनुहोस्:

TRUE
वा
FALSE

व्याख्या नदिनुहोस्।
TRUE वा FALSE बाहेक अरू केही नलेख्नुहोस्।

कथन:
{statement}

उत्तर:
"""


# ================================================================
# 13. ONE EVALUATION
# ================================================================

def run_one(
    item_id,
    language,
    row
):

    if language == "English":

        statement = str(
            row["en"]
        )

        prompt = build_english_prompt(
            statement
        )

        reference = row["Nepali"]

    else:

        statement = str(
            row["Nepali"]
        )

        prompt = build_nepali_prompt(
            statement
        )

        reference = row["en"]


    # First model call.
    raw_output = ask_gemma(
        prompt
    )

    prediction = normalize_prediction(
        raw_output
    )


    # One retry ONLY if malformed.
    # Never loop forever.
    if prediction == "NON_STANDARD":

        retry_prompt = f"""
Answer this factual statement.

Your response MUST be exactly one word:
TRUE
or
FALSE

No explanation.

Statement:
{statement}

Answer:
"""

        retry_raw = ask_gemma(
            retry_prompt
        )

        retry_prediction = normalize_prediction(
            retry_raw
        )

        if retry_prediction != "NON_STANDARD":

            raw_output = retry_raw
            prediction = retry_prediction


    # ============================================================
    # OUR EVALUATION
    # ============================================================
    #
    # These fields are NOT generated by Gemma.
    # They are determined by our evaluation methodology.
    # ============================================================

    gold = str(
        row["GOLD"]
    ).strip().upper()


    if prediction == "NON_STANDARD":

        verdict = "Incorrect"

        error_type = (
            "Garbage/Non-standard output"
        )

        strict_error = True


    elif prediction == gold:

        verdict = "Correct"

        error_type = "None"

        strict_error = False


    else:

        verdict = "Incorrect"

        error_type = (
            "Wrong TRUE/FALSE prediction"
        )

        strict_error = True


    return {
        "ID": item_id,
        "Language": language,
        "Statement": statement,
        "Reference Nepali": reference,
        "Gold Label": gold,

        # Model output:
        "Gemma Raw Output": raw_output,
        "Gemma Answer": prediction,

        # Our evaluation:
        "Verdict": verdict,
        "Error Type": error_type,
        "Strict Error": strict_error
    }


# ================================================================
# 14. MAIN LOOP
# ================================================================
#
# Every cycle recalculates missing pairs from the actual CSV.
# Therefore real gaps are automatically recovered.
# ================================================================

while True:

    # Reload file = source of truth.
    results = pd.read_csv(
        OUTPUT_FILE
    )

    completed = get_pairs(
        results
    )

    missing_pairs = sorted(
        expected_pairs - completed,
        key=lambda x: (
            int(float(x[0])),
            0 if x[1] == "English" else 1
        )
    )


    # ------------------------------------------------------------
    # FINISHED
    # ------------------------------------------------------------

    if not missing_pairs:

        print(
            "\n✅ ALL 12,002 EVALUATIONS COMPLETE"
        )

        break


    # ------------------------------------------------------------
    # EXACT FIRST MISSING PAIR
    # ------------------------------------------------------------

    item_id, language = missing_pairs[0]

    row = source_lookup[item_id]


    # ------------------------------------------------------------
    # RUN GEMMA
    # ------------------------------------------------------------

    new_row = run_one(
        item_id,
        language,
        row
    )


    # ------------------------------------------------------------
    # DUPLICATE CHECK
    # ------------------------------------------------------------

    results = pd.read_csv(
        OUTPUT_FILE
    )

    if (
        item_id,
        language
    ) in get_pairs(results):

        print(
            f"{item_id} | {language} "
            "already saved. Continuing."
        )

        continue


    # ------------------------------------------------------------
    # SAVE RESULT
    # ------------------------------------------------------------

    results = pd.concat(
        [
            results,
            pd.DataFrame([new_row])
        ],
        ignore_index=True
    )

    safe_save(
        results
    )


    # ------------------------------------------------------------
    # VERIFY
    # ------------------------------------------------------------

    verified = pd.read_csv(
        OUTPUT_FILE
    )

    verified_pairs = get_pairs(
        verified
    )

    if (
        item_id,
        language
    ) not in verified_pairs:

        raise RuntimeError(
            f"Save verification failed: "
            f"{item_id} | {language}"
        )


    remaining = (
        expected_pairs - verified_pairs
    )


    print(
        f"{item_id} | {language} | "
        f"Gemma={new_row['Gemma Answer']} | "
        f"Verdict={new_row['Verdict']} | "
        f"Error={new_row['Error Type']} | "
        f"Saved={len(verified_pairs)} | "
        f"Remaining={len(remaining)}"
    )


# ================================================================
# 15. FINAL AUDIT + AUTHORITATIVE EVALUATION
# ================================================================

final_df = pd.read_csv(
    OUTPUT_FILE
)

final_pairs = get_pairs(
    final_df
)

missing_final = (
    expected_pairs - final_pairs
)

duplicate_rows = int(
    final_df.duplicated(
        subset=["ID", "Language"],
        keep=False
    ).sum()
)


# Recalculate our evaluation fields from scratch.
# Gemma is NOT trusted for these fields.

verdicts = []
error_types = []
strict_errors = []

for _, row in final_df.iterrows():

    gold = str(
        row["Gold Label"]
    ).strip().upper()

    answer = str(
        row["Gemma Answer"]
    ).strip().upper()


    if answer == "NON_STANDARD":

        verdicts.append("Incorrect")
        error_types.append(
            "Garbage/Non-standard output"
        )
        strict_errors.append(True)


    elif answer == gold:

        verdicts.append("Correct")
        error_types.append("None")
        strict_errors.append(False)


    else:

        verdicts.append("Incorrect")
        error_types.append(
            "Wrong TRUE/FALSE prediction"
        )
        strict_errors.append(True)


final_df["Verdict"] = verdicts
final_df["Error Type"] = error_types
final_df["Strict Error"] = strict_errors


# Save authoritative evaluation fields.
safe_save(
    final_df
)


# ================================================================
# 16. FINAL VERIFICATION
# ================================================================

final_df = pd.read_csv(
    OUTPUT_FILE
)

final_pairs = get_pairs(
    final_df
)

invalid_answers = int(
    (
        ~final_df["Gemma Answer"].isin(
            [
                "TRUE",
                "FALSE",
                "NON_STANDARD"
            ]
        )
    ).sum()
)

duplicate_rows = int(
    final_df.duplicated(
        subset=["ID", "Language"],
        keep=False
    ).sum()
)

missing_final = (
    expected_pairs - final_pairs
)


print("\n" + "=" * 70)
print("FINAL POLY-FEVER AUDIT")
print("=" * 70)

print(
    "Expected evaluations:",
    len(expected_pairs)
)

print(
    "Actual evaluations:",
    len(final_pairs)
)

print(
    "Missing pairs:",
    len(missing_final)
)

print(
    "Duplicate rows:",
    duplicate_rows
)

print(
    "Invalid stored answers:",
    invalid_answers
)

print("\nLanguage counts:")
print(
    final_df["Language"].value_counts()
)

print("\nVerdict counts:")
print(
    final_df["Verdict"].value_counts()
)

print("\nError type counts:")
print(
    final_df["Error Type"].value_counts()
)


if (
    len(missing_final) == 0
    and duplicate_rows == 0
    and invalid_answers == 0
):

    print(
        "\n✅ POLY-FEVER COMPLETE AND VERIFIED."
    )

else:

    print(
        "\n❌ NOT COMPLETE."
    )

    print(
        "First missing pairs:",
        list(missing_final)[:30]
    )


print(
    "\nFINAL FILE:"
)

print(
    OUTPUT_FILE
)

Folder: C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research
Model: gemma3:1b
Output: C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research\Poly_FEVER_Gemma3_1B_QA_FINAL.csv

SOURCE:
C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research\Hexagon-Dataset-Nepali-Poly-FEVER_PAIRED_FULL_COLUMNS (1).csv

SOURCE VERIFIED
Rows: 6001
Expected evaluations: 12002
Expected pairs: 12002

Existing clean result found.
Resuming from saved file.

✅ Ollama online.
4483 | Nepali | Gemma=TRUE | Verdict=Correct | Error=None | Saved=4968 | Remaining=7034
4484 | English | Gemma=FALSE | Verdict=Incorrect | Error=Wrong TRUE/FALSE prediction | Saved=4969 | Remaining=7033
4484 | Nepali | Gemma=FALSE | Verdict=Incorrect | Error=Wrong TRUE/FALSE prediction | Saved=4970 | Remaining=7032
4485 | English | Gemma=TRUE | Verdict=Correct | Error=None | Saved=4971 | Remaining=7031


KeyboardInterrupt: 

In [2]:
# ================================================================
# POLY-FEVER — RESUME FROM EXISTING QA CHECKPOINT
# ================================================================

import os
import re
import time
from pathlib import Path

import pandas as pd
import requests


# ================================================================
# 1. SETTINGS
# ================================================================

FOLDER = Path(
    r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research"
)

SOURCE_FILE = FOLDER / (
    "Hexagon-Dataset-Nepali-Poly-FEVER_PAIRED_FULL_COLUMNS (1).csv"
)

OUTPUT_FILE = FOLDER / (
    "Poly_FEVER_Gemma3_1B_QA_FINAL.csv"
)

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:1b"


# ================================================================
# 2. LOAD SOURCE
# ================================================================

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        f"Source file not found:\n{SOURCE_FILE}"
    )

df = pd.read_csv(SOURCE_FILE)

df["ID_STR"] = (
    df["ID"]
    .astype(str)
    .str.strip()
)

df["GOLD"] = (
    df["Label"]
    .astype(str)
    .str.strip()
    .str.upper()
)

required = ["ID", "en", "Nepali", "Label"]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing source columns: {missing}"
    )

if not set(df["GOLD"]).issubset({"TRUE", "FALSE"}):
    raise ValueError(
        "Source Gold Label contains values other than TRUE/FALSE."
    )

print("Source rows:", len(df))
print("Expected evaluations:", len(df) * 2)


# ================================================================
# 3. LOAD EXISTING CHECKPOINT
# ================================================================

if not OUTPUT_FILE.exists():
    raise FileNotFoundError(
        f"Your saved checkpoint was not found:\n{OUTPUT_FILE}"
    )

results = pd.read_csv(OUTPUT_FILE)

required_result = [
    "ID",
    "Language",
    "Statement",
    "Reference Nepali",
    "Gold Label",
    "Gemma Raw Output",
    "Gemma Answer",
    "Verdict",
    "Error Type",
    "Strict Error"
]

missing_result = [
    c for c in required_result
    if c not in results.columns
]

if missing_result:
    raise ValueError(
        f"Checkpoint is missing columns: {missing_result}"
    )

# Remove accidental duplicate pairs only.
results = results.drop_duplicates(
    subset=["ID", "Language"],
    keep="last"
).reset_index(drop=True)

print("Existing saved evaluations:", len(results))
print(
    results["Language"]
    .value_counts()
    .to_dict()
)


# ================================================================
# 4. EXPECTED PAIRS
# ================================================================

expected_pairs = {
    (str(item_id), language)
    for item_id in df["ID_STR"]
    for language in ["English", "Nepali"]
}


def get_pairs(data):
    return {
        (
            str(row["ID"]).strip(),
            str(row["Language"]).strip()
        )
        for _, row in data.iterrows()
    }


completed_pairs = get_pairs(results)

missing_pairs = sorted(
    expected_pairs - completed_pairs,
    key=lambda x: (
        int(float(x[0])),
        0 if x[1] == "English" else 1
    )
)

print("Completed pairs:", len(completed_pairs))
print("Missing pairs:", len(missing_pairs))

if missing_pairs:
    print("First missing pairs:", missing_pairs[:20])


# ================================================================
# 5. OLLAMA CHECK
# ================================================================

def wait_for_ollama():

    while True:

        try:

            r = requests.get(
                "http://localhost:11434/api/tags",
                timeout=10
            )

            if r.status_code == 200:
                print("✅ Ollama online.")
                return

        except Exception:
            pass

        print("Ollama unavailable. Retrying in 10 seconds...")
        time.sleep(10)


wait_for_ollama()


# ================================================================
# 6. SAVE SAFELY
# ================================================================

def safe_save(data):

    temp_file = OUTPUT_FILE.with_suffix(
        OUTPUT_FILE.suffix + ".tmp"
    )

    while True:

        try:

            data.to_csv(
                temp_file,
                index=False,
                encoding="utf-8-sig"
            )

            os.replace(
                temp_file,
                OUTPUT_FILE
            )

            return

        except PermissionError:

            print(
                "CSV is open/locked. "
                "Close it and retrying in 5 seconds..."
            )

            time.sleep(5)


# ================================================================
# 7. GEMMA REQUEST
# ================================================================

def ask_gemma(prompt):

    retry = 0
    wait = 3

    while True:

        try:

            r = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt,
                    "stream": False,
                    "keep_alive": "5m",
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=(15, 600)
            )

            if r.status_code == 200:

                return r.json().get(
                    "response",
                    ""
                ).strip()

            print(
                "Ollama HTTP error:",
                r.status_code
            )

        except requests.exceptions.ConnectionError:

            print("Ollama connection lost.")

        except requests.exceptions.Timeout:

            print("Ollama timeout.")

        except Exception as e:

            print("Ollama error:", e)

        retry += 1

        print(
            f"Retry {retry} in {wait}s..."
        )

        time.sleep(wait)

        wait = min(
            wait * 2,
            30
        )


# ================================================================
# 8. NORMALIZE MODEL OUTPUT
# ================================================================

def normalize_prediction(text):

    text = str(text).strip().upper()

    if text == "TRUE":
        return "TRUE"

    if text == "FALSE":
        return "FALSE"

    first = (
        text.split()[0]
        if text.split()
        else ""
    )

    if first == "TRUE":
        return "TRUE"

    if first == "FALSE":
        return "FALSE"

    if (
        "TRUE" in text
        and "FALSE" not in text
    ):
        return "TRUE"

    if (
        "FALSE" in text
        and "TRUE" not in text
    ):
        return "FALSE"

    return "NON_STANDARD"


# ================================================================
# 9. ONE TARGET
# ================================================================

def evaluate_target(
    item_id,
    language,
    row
):

    if language == "English":

        statement = str(
            row["en"]
        )

        prompt = f"""
You are a factual verification model.

Classify the following statement using your existing knowledge.

Return exactly one label:
TRUE
or
FALSE

Do not explain.
Do not output anything except TRUE or FALSE.

Statement:
{statement}

Answer:
"""

        reference = row["Nepali"]

    else:

        statement = str(
            row["Nepali"]
        )

        prompt = f"""
तपाईं एउटा तथ्यात्मक कथनको सत्यता जाँच गर्ने मोडेल हुनुहुन्छ।

तलको कथन TRUE वा FALSE भनेर वर्गीकरण गर्नुहोस्।

ठ्याक्कै एउटा मात्र उत्तर दिनुहोस्:

TRUE
वा
FALSE

व्याख्या नदिनुहोस्।
TRUE वा FALSE बाहेक अरू केही नलेख्नुहोस्।

कथन:
{statement}

उत्तर:
"""

        reference = row["en"]


    raw_output = ask_gemma(
        prompt
    )

    answer = normalize_prediction(
        raw_output
    )

    # One retry only for malformed output.
    if answer == "NON_STANDARD":

        retry_prompt = f"""
Answer the following factual statement.

Your response MUST be exactly one word:

TRUE

or

FALSE

Nothing else.

Statement:
{statement}

Answer:
"""

        retry_raw = ask_gemma(
            retry_prompt
        )

        retry_answer = normalize_prediction(
            retry_raw
        )

        if retry_answer != "NON_STANDARD":

            raw_output = retry_raw
            answer = retry_answer


    # ============================================================
    # OUR EVALUATION — NOT GEMMA
    # ============================================================

    gold = str(
        row["GOLD"]
    ).upper()

    if answer == "NON_STANDARD":

        verdict = "Incorrect"
        error_type = "Garbage/Non-standard output"
        strict_error = True

    elif answer == gold:

        verdict = "Correct"
        error_type = "None"
        strict_error = False

    else:

        verdict = "Incorrect"
        error_type = "Wrong TRUE/FALSE prediction"
        strict_error = True


    return {
        "ID": item_id,
        "Language": language,
        "Statement": statement,
        "Reference Nepali": reference,
        "Gold Label": gold,
        "Gemma Raw Output": raw_output,
        "Gemma Answer": answer,
        "Verdict": verdict,
        "Error Type": error_type,
        "Strict Error": strict_error
    }


# ================================================================
# 10. RESUME LOOP
# ================================================================

while True:

    # Reload checkpoint every iteration.
    results = pd.read_csv(
        OUTPUT_FILE
    )

    completed_pairs = get_pairs(
        results
    )

    # Recalculate gaps from the REAL saved CSV.
    missing_pairs = sorted(
        expected_pairs - completed_pairs,
        key=lambda x: (
            int(float(x[0])),
            0 if x[1] == "English" else 1
        )
    )

    # ------------------------------------------------------------
    # COMPLETE
    # ------------------------------------------------------------

    if not missing_pairs:

        print("\n" + "=" * 70)
        print("✅ ALL 12,002 EVALUATIONS COMPLETE")
        print("=" * 70)

        break

    # ------------------------------------------------------------
    # FIRST REAL MISSING PAIR
    # ------------------------------------------------------------

    item_id, language = missing_pairs[0]

    row = df[
        df["ID_STR"] == item_id
    ].iloc[0]

    print(
        f"\nProcessing: {item_id} | {language}"
    )

    # ------------------------------------------------------------
    # GEMMA
    # ------------------------------------------------------------

    new_row = evaluate_target(
        item_id,
        language,
        row
    )

    # ------------------------------------------------------------
    # RECHECK BEFORE SAVE
    # ------------------------------------------------------------

    results = pd.read_csv(
        OUTPUT_FILE
    )

    current_pairs = get_pairs(
        results
    )

    if (
        item_id,
        language
    ) in current_pairs:

        print(
            f"{item_id} | {language} already saved."
        )

        continue

    # ------------------------------------------------------------
    # SAVE
    # ------------------------------------------------------------

    results = pd.concat(
        [
            results,
            pd.DataFrame([new_row])
        ],
        ignore_index=True
    )

    safe_save(
        results
    )

    # ------------------------------------------------------------
    # VERIFY
    # ------------------------------------------------------------

    verified = pd.read_csv(
        OUTPUT_FILE
    )

    verified_pairs = get_pairs(
        verified
    )

    if (
        item_id,
        language
    ) not in verified_pairs:

        raise RuntimeError(
            f"Save verification failed: "
            f"{item_id} | {language}"
        )

    print(
        f"{item_id} | {language} | "
        f"Gemma={new_row['Gemma Answer']} | "
        f"Verdict={new_row['Verdict']} | "
        f"Error={new_row['Error Type']} | "
        f"Saved={len(verified_pairs)} | "
        f"Remaining={len(expected_pairs - verified_pairs)}"
    )


# ================================================================
# 11. FINAL AUDIT
# ================================================================

final_df = pd.read_csv(
    OUTPUT_FILE
)

final_pairs = get_pairs(
    final_df
)

missing_final = expected_pairs - final_pairs

duplicates = int(
    final_df.duplicated(
        subset=["ID", "Language"],
        keep=False
    ).sum()
)

print("\n" + "=" * 70)
print("FINAL AUDIT")
print("=" * 70)

print("Expected:", len(expected_pairs))
print("Actual:", len(final_pairs))
print("Missing:", len(missing_final))
print("Duplicate rows:", duplicates)

print("\nLanguage counts:")
print(
    final_df["Language"].value_counts()
)

print("\nVerdict counts:")
print(
    final_df["Verdict"].value_counts()
)

print("\nError type counts:")
print(
    final_df["Error Type"].value_counts()
)

if (
    len(missing_final) == 0
    and duplicates == 0
):

    print(
        "\n✅ COMPLETE AND VERIFIED"
    )

else:

    print(
        "\n❌ STILL INCOMPLETE"
    )

    print(
        "First missing:",
        list(missing_final)[:30]
    )

print("\nFile:")
print(OUTPUT_FILE)

Source rows: 6001
Expected evaluations: 12002
Existing saved evaluations: 7832
{'English': 3916, 'Nepali': 3916}
Completed pairs: 7832
Missing pairs: 4170
First missing pairs: [('6916', 'English'), ('6916', 'Nepali'), ('6917', 'English'), ('6917', 'Nepali'), ('6918', 'English'), ('6918', 'Nepali'), ('6919', 'English'), ('6919', 'Nepali'), ('6920', 'English'), ('6920', 'Nepali'), ('6921', 'English'), ('6921', 'Nepali'), ('6922', 'English'), ('6922', 'Nepali'), ('6923', 'English'), ('6923', 'Nepali'), ('6924', 'English'), ('6924', 'Nepali'), ('6925', 'English'), ('6925', 'Nepali')]
✅ Ollama online.

Processing: 6916 | English
6916 | English | Gemma=FALSE | Verdict=Correct | Error=None | Saved=7833 | Remaining=4169

Processing: 6916 | Nepali
6916 | Nepali | Gemma=TRUE | Verdict=Incorrect | Error=Wrong TRUE/FALSE prediction | Saved=7834 | Remaining=4168

Processing: 6917 | English
6917 | English | Gemma=TRUE | Verdict=Incorrect | Error=Wrong TRUE/FALSE prediction | Saved=7835 | Remaining=4